# CrossMod-Transformer

## 1. Imports

In [ ]:
import os
import time
import math
import random
import pickle

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut


## 2. Configuration


In [ ]:
# ── User-facing config ──────────────────────────────────────────────────────
BASE_DIR   = "./data"
MODEL      = "cmt"        # "fcn" | "alstm" | "fat" | "cmt"
MODALITY   = None          # "eda" | "ecg"  (None for CMT, which uses both)
CV         = "loso"        # "loso" | "10fold"
ONLY_FOLD  = None          # run a single fold (e.g. 3), or None for all folds

# ── Training hyper-parameters ───────────────────────────────────────────────
LR              = 1e-5
BATCH_SIZE      = 128
EPOCHS          = 100
SCALER_TYPE     = "robust"   # "robust" | "power"
SEQ_LEN         = 138

# ── ALSTM ───────────────────────────────────────────────────────────────────
LSTM_HIDDEN     = 128
LSTM_LAYERS     = 2

# ── Transformer (FAT / CMT) ─────────────────────────────────────────────────
N_HEADS         = 8
TRANS_HIDDEN    = 128
TRANS_LAYERS    = 2

# ── LR scheduler ────────────────────────────────────────────────────────────
SCHED_MILESTONES = [500]
SCHED_GAMMA      = 0.001

# ── Misc ─────────────────────────────────────────────────────────────────────
USE_CUDA    = True
CLASS_NAMES = ["BLN", "PA4"]


In [ ]:
def build_config():
    if not os.path.exists(BASE_DIR):
        raise ValueError(f"BASE_DIR does not exist: {BASE_DIR}")

    if MODEL != "cmt" and MODALITY is None:
        raise ValueError("Set MODALITY to 'eda' or 'ecg' for non-CMT models.")
    if MODEL == "cmt" and MODALITY is not None:
        raise ValueError("MODALITY must be None for CMT (it uses both EDA and ECG).")

    def feature_dir(mod, arch):
        return os.path.join(BASE_DIR, f"{mod.upper()}_{arch.upper()}")

    cfg = {
        "model": MODEL,
        "modality": MODALITY,
        "cv": CV,
        "only_fold": ONLY_FOLD,

        "lr": LR,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,

        "data_path": os.path.join(BASE_DIR, "data.pkl"),
        "scaler": SCALER_TYPE,
        "seq_len": SEQ_LEN,
        "class_names": CLASS_NAMES,

        "lstm_params": {
            "input_channels": 1,
            "hidden_size": LSTM_HIDDEN,
            "num_layers": LSTM_LAYERS,
            "alstm_out_dim": 128,
            "num_classes": 2,
            "dropout": 0.3,
        },

        "fcn_params": {
            "input_channels": 1,
            "out_filters": 128,
            "dropout": 0.17,
        },

        "fat_params": {
            "n_heads": N_HEADS,
            "hidden_dim": TRANS_HIDDEN,
            "n_layers": TRANS_LAYERS,
            "num_classes": 2,
            "out_dim": 256,
        },

        "cmt_params": {
            "n_heads": N_HEADS,
            "hidden_dim": TRANS_HIDDEN,
            "n_layers": TRANS_LAYERS,
            "num_classes": 2,
        },

        "sched_milestones": SCHED_MILESTONES,
        "sched_gamma": SCHED_GAMMA,

        "feature_dirs": {
            "EDA_FCN":   feature_dir("eda", "fcn"),
            "EDA_ALSTM": feature_dir("eda", "alstm"),
            "EDA_FAT":   feature_dir("eda", "fat"),
            "ECG_FCN":   feature_dir("ecg", "fcn"),
            "ECG_ALSTM": feature_dir("ecg", "alstm"),
            "ECG_FAT":   feature_dir("ecg", "fat"),
            "CMT":       os.path.join(BASE_DIR, "CMT"),
        },

        "use_cuda": USE_CUDA,
    }
    return cfg

cfg = build_config()
print("Config ready. Model:", cfg["model"])


## 3. Datasets




In [ ]:
def linear_resample(signal, target_len):
    src_idx = np.linspace(0, len(signal) - 1, num=len(signal))
    dst_idx = np.linspace(0, len(signal) - 1, num=target_len)
    return np.interp(dst_idx, src_idx, signal)


class RawSignalDataset(Dataset):

    def __init__(self, df, modality, split, seq_len, subject_ids, scaler=None):
        self.modality = modality.lower()
        self.seq_len  = seq_len

        subset = df[df["subject_id"].isin(subject_ids)].copy()
        subset = subset.sort_values(["series_id", "time"])

        self.scaler = scaler or RobustScaler()
        col = f"{self.modality}_norm"
        if split == "train":
            subset[col] = self.scaler.fit_transform(subset[[self.modality]])
        else:
            subset[col] = self.scaler.transform(subset[[self.modality]])

        self.samples = []
        for _, group in subset.groupby("series_id"):
            values = group[col].values
            if len(values) < seq_len:
                continue
            resampled = linear_resample(values, seq_len)
            label = int(np.bincount(group["class_id"].values).argmax())
            signal_tensor = torch.tensor(resampled, dtype=torch.float).unsqueeze(1)  # [S, 1]
            self.samples.append((torch.tensor(label, dtype=torch.long), signal_tensor))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]


def prepare_raw_datasets(df, modality, seq_len, train_ids, test_ids):
    train_ds = RawSignalDataset(df, modality, "train", seq_len, train_ids)
    test_ds  = RawSignalDataset(df, modality, "test",  seq_len, test_ids,
                                scaler=train_ds.scaler)
    return train_ds, test_ds


In [ ]:
class FusionFeatureDataset(Dataset):

    def __init__(self, feat_a, feat_b, labels):
        assert len(feat_a) == len(feat_b) == len(labels), "Length mismatch in feature sets"
        self.feat_a  = torch.tensor(feat_a, dtype=torch.float)
        self.feat_b  = torch.tensor(feat_b, dtype=torch.float)
        self.labels  = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.feat_a[idx], self.feat_b[idx], self.labels[idx]


def load_fusion_dataset(dir_a, dir_b, arch_a, arch_b, fold):
    def npy(d, arch, split):
        return np.load(os.path.join(d, f"{arch}_features_{split}_fold{fold}.npy"))

    def labels(d, split):
        return np.load(os.path.join(d, f"labels_{split}_fold{fold}.npy"))

    train_ds = FusionFeatureDataset(npy(dir_a, arch_a, "train"),
                                    npy(dir_b, arch_b, "train"),
                                    labels(dir_a, "train"))
    test_ds  = FusionFeatureDataset(npy(dir_a, arch_a, "test"),
                                    npy(dir_b, arch_b, "test"),
                                    labels(dir_a, "test"))
    return train_ds, test_ds


## 4. Model Definitions

### 4.1 Shared Transformer building blocks



In [ ]:
class FourierPositionalEncoding(nn.Module):

    def __init__(self, d_model, base=10000.0):
        super().__init__()
        self.d_model = d_model
        self.base    = base

    def forward(self, x):
        B, T, D = x.size()
        pos      = torch.arange(T, device=x.device).unsqueeze(1).float()
        freq     = torch.exp(
            torch.arange(0, D, 2, device=x.device).float() * -(math.log(self.base) / D)
        )
        pe = torch.zeros(T, D, device=x.device)
        pe[:, 0::2] = torch.sin(pos * freq)
        pe[:, 1::2] = torch.cos(pos * freq)
        return x + pe.unsqueeze(0).expand(B, -1, -1)


class TransformerEncoderLayer(nn.Module):

    def __init__(self, dim, n_heads, ff_dim):
        super().__init__()
        self.attn   = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.ff     = nn.Sequential(nn.Linear(dim, ff_dim), nn.ReLU(), nn.Linear(ff_dim, dim))
        self.norm1  = nn.LayerNorm(dim)
        self.norm2  = nn.LayerNorm(dim)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        x = self.norm2(x + self.ff(x))
        return x


class TransformerEncoderBlock(nn.Module):

    def __init__(self, dim, n_heads, ff_dim, n_layers):
        super().__init__()
        self.pos_enc = FourierPositionalEncoding(dim)
        self.layers  = nn.Sequential(*[
            TransformerEncoderLayer(dim, n_heads, ff_dim) for _ in range(n_layers)
        ])

    def forward(self, x):
        return self.layers(self.pos_enc(x))


### 4.2 FCN — Fully Convolutional Network

In [ ]:
class FCNModel(nn.Module):

    def __init__(self, input_channels, out_filters, dropout=0.2718491):
        super().__init__()
        F = out_filters

        self.block1 = nn.Sequential(
            nn.Conv1d(input_channels, F,     kernel_size=8, padding="same"),
            nn.BatchNorm1d(F), nn.ReLU(), nn.Dropout(dropout)
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(F, F * 2,              kernel_size=5, padding="same"),
            nn.BatchNorm1d(F * 2), nn.ReLU(), nn.Dropout(dropout)
        )
        self.block3 = nn.Sequential(
            nn.Conv1d(F * 2, F,              kernel_size=3, padding="same"),
            nn.BatchNorm1d(F), nn.ReLU(), nn.Dropout(dropout)
        )
        self.gap       = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(F, 2)

    def forward(self, x):
        x        = self.block1(x)
        x        = self.block2(x)
        x        = self.block3(x)           # [B, F, S]
        features = x.permute(0, 2, 1)       # [B, S, F]  ← used by FAT
        pooled   = self.gap(x).squeeze(-1)  # [B, F]
        return self.classifier(pooled), features


### 4.3 ALSTM — Attention LSTM

In [ ]:
def elish(x):
    return torch.where(x >= 0, x, (torch.exp(x) - 1) / (1 + torch.exp(-x)))


class ELiSHAttention(nn.Module):

    def __init__(self, hidden_size):
        super().__init__()
        self.score = nn.Linear(hidden_size, 1)

    def forward(self, hidden_states):
        # hidden_states: [B, S, H]
        weights  = F.softmax(elish(self.score(hidden_states).squeeze(-1)), dim=1)  # [B, S]
        context  = torch.bmm(weights.unsqueeze(1), hidden_states).squeeze(1)       # [B, H]
        return context


class ALSTMModel(nn.Module):

    def __init__(self, input_channels, hidden_size, num_layers,
                 num_classes, dropout=0.389102478, alstm_out_dim=128):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        self.lstm      = nn.LSTM(input_channels, hidden_size, num_layers, batch_first=True)
        self.dropout   = nn.Dropout(dropout)
        self.bn        = nn.BatchNorm1d(hidden_size)
        self.attention = ELiSHAttention(hidden_size)
        self.classifier = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        x  = x.permute(0, 2, 1)                                # [B, C, S] → [B, S, C]
        B  = x.size(0)
        h0 = torch.zeros(self.num_layers, B, self.hidden_size, device=x.device)
        c0 = torch.zeros(self.num_layers, B, self.hidden_size, device=x.device)

        out, _ = self.lstm(x, (h0, c0))                        # [B, S, H]
        out    = self.dropout(out)

        # BatchNorm expects [B*S, H], then reshape back
        normed   = self.bn(out.contiguous().view(-1, self.hidden_size))
        features = normed.view(B, -1, self.hidden_size)         # [B, S, H] ← used by FAT

        context  = self.attention(features)                     # [B, H]
        combined = torch.cat([context, features[:, -1, :]], dim=1)  # [B, 2H]
        return self.classifier(combined), features


### 4.4 FAT — FCN + ALSTM + Transformer fusion

In [ ]:
class FCNALSTMTransformer(nn.Module):

    def __init__(self, fcn_dim, lstm_dim, n_heads, hidden_dim,
                 n_layers, num_classes, out_dim):
        super().__init__()
        self.fcn_encoder  = TransformerEncoderBlock(fcn_dim,  n_heads, hidden_dim, n_layers)
        self.lstm_encoder = TransformerEncoderBlock(lstm_dim, n_heads, hidden_dim, n_layers)

        self.cross_fcn  = nn.MultiheadAttention(lstm_dim, n_heads, batch_first=True)
        self.cross_lstm = nn.MultiheadAttention(lstm_dim, n_heads, batch_first=True)

        self.head = nn.Sequential(
            nn.Linear(lstm_dim * 2, 512), nn.ReLU(),
            nn.Linear(512, 256),          nn.ReLU(),
            nn.Linear(256, out_dim),      nn.ReLU(),
            nn.Linear(out_dim, num_classes),
        )

    def forward(self, fcn_feat, lstm_feat):
        if fcn_feat.size(1) != lstm_feat.size(1):
            raise ValueError("FCN and ALSTM features must share the same sequence length.")

        enc_fcn  = self.fcn_encoder(fcn_feat)
        enc_lstm = self.lstm_encoder(lstm_feat)

        cross_f, _ = self.cross_fcn( enc_fcn,  enc_lstm, enc_lstm)
        cross_l, _ = self.cross_lstm(enc_lstm, enc_fcn,  enc_fcn)

        fused  = torch.cat([cross_f, cross_l], dim=2)   # [B, S, lstm_dim*2]
        pooled = fused.mean(dim=1)
        return self.head(pooled), fused


### 4.5 CrossMod-Transformer — EDA × ECG cross-modal fusion

In [ ]:
class CrossModTransformer(nn.Module):

    def __init__(self, eda_dim, ecg_dim, n_heads, hidden_dim,
                 n_layers, num_classes):
        super().__init__()
        self.eda_proj     = nn.Linear(eda_dim, ecg_dim)   # align dims
        self.eda_encoder  = TransformerEncoderBlock(eda_dim, n_heads, hidden_dim, n_layers)
        self.ecg_encoder  = TransformerEncoderBlock(ecg_dim, n_heads, hidden_dim, n_layers)

        self.cross_eda = nn.MultiheadAttention(ecg_dim, n_heads, batch_first=True)
        self.cross_ecg = nn.MultiheadAttention(ecg_dim, n_heads, batch_first=True)

        self.head = nn.Sequential(
            nn.Linear(ecg_dim * 2, 512), nn.ReLU(),
            nn.Linear(512, 256),         nn.ReLU(),
            nn.Linear(256, 128),         nn.ReLU(),
            nn.Linear(128, num_classes),
        )

    def forward(self, eda_feat, ecg_feat):
        if eda_feat.size(1) != ecg_feat.size(1):
            raise ValueError("EDA and ECG features must share the same sequence length.")

        enc_eda = self.eda_proj(self.eda_encoder(eda_feat))
        enc_ecg = self.ecg_encoder(ecg_feat)

        cross_e, _ = self.cross_eda(enc_eda, enc_ecg, enc_ecg)
        cross_g, _ = self.cross_ecg(enc_ecg, enc_eda, enc_eda)

        fused  = torch.cat([cross_e, cross_g], dim=2)    # [B, S, ecg_dim*2]
        pooled = fused.mean(dim=1)
        return self.head(pooled), fused


### 4.6 Model factory

In [ ]:
def build_model(cfg, device):
    name = cfg["model"]

    if name == "fcn":
        model = FCNModel(**cfg["fcn_params"])

    elif name == "alstm":
        p     = cfg["lstm_params"]
        model = ALSTMModel(
            input_channels=p["input_channels"],
            hidden_size=p["hidden_size"],
            num_layers=p["num_layers"],
            num_classes=p["num_classes"],
            dropout=p["dropout"],
            alstm_out_dim=p["alstm_out_dim"],
        )

    elif name == "fat":
        p     = cfg["fat_params"]
        model = FCNALSTMTransformer(
            fcn_dim=cfg["fcn_params"]["out_filters"],
            lstm_dim=cfg["lstm_params"]["alstm_out_dim"],
            n_heads=p["n_heads"],
            hidden_dim=p["hidden_dim"],
            n_layers=p["n_layers"],
            num_classes=p["num_classes"],
            out_dim=p["out_dim"],
        )

    elif name == "cmt":
        p     = cfg["cmt_params"]
        model = CrossModTransformer(
            eda_dim=cfg["fat_params"]["out_dim"],
            ecg_dim=cfg["fat_params"]["out_dim"],
            n_heads=p["n_heads"],
            hidden_dim=p["hidden_dim"],
            n_layers=p["n_layers"],
            num_classes=p["num_classes"],
        )

    else:
        raise ValueError(f"Unknown model: {name}")

    return model.to(device)


## 5. Training & Validation

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device,
                    mode, model_name, save_features=False):
    model.train()
    total_loss, total_correct = 0.0, 0
    collected_features, collected_labels = [], []

    for batch in tqdm(loader, desc="  train", leave=False):

        if mode == "raw":
            labels, signals = batch
            labels  = labels.long().to(device)
            signals = signals.to(device)
            if signals.ndim == 2:
                signals = signals.unsqueeze(1)
            elif signals.ndim == 3:
                signals = signals.permute(0, 2, 1)
            logits, *rest = model(signals)

        elif mode == "feature":
            if model_name == "cmt":
                feat_a, feat_b, labels = batch
                feat_a, feat_b = feat_a.to(device), feat_b.to(device)
                labels = labels.to(device)
                logits, *rest = model(feat_a, feat_b)
            else:
                feat_a, feat_b, labels = batch
                feat_a, feat_b = feat_a.to(device), feat_b.to(device)
                labels = labels.to(device)
                logits, *rest = model(feat_a, feat_b)
        else:
            raise ValueError(f"mode must be 'raw' or 'feature', got: {mode}")

        loss = criterion(logits, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss    += loss.item()
        total_correct += (logits.argmax(dim=1) == labels).sum().item()

        if save_features and rest:
            collected_features.append(rest[0].detach().cpu().numpy())
            collected_labels.append(labels.detach().cpu().numpy())

    n          = len(loader.dataset)
    epoch_loss = total_loss / n
    epoch_acc  = 100.0 * total_correct / n

    if save_features:
        return epoch_loss, epoch_acc, collected_features, collected_labels
    return epoch_loss, epoch_acc


In [ ]:
def evaluate(model, loader, criterion, device, mode, model_name):
    model.eval()
    total_loss, total_correct = 0.0, 0
    conf_matrix  = torch.zeros(2, 2)
    all_preds, all_logits, all_labels, all_features = [], [], [], []

    with torch.no_grad():
        t0 = time.time()

        for batch in tqdm(loader, desc="  eval ", leave=False):

            if mode == "raw":
                labels, signals = batch
                labels  = labels.long().to(device)
                signals = signals.to(device)
                if signals.ndim == 2:
                    signals = signals.unsqueeze(1)
                elif signals.ndim == 3:
                    signals = signals.permute(0, 2, 1)
                logits, *rest = model(signals)

            elif mode == "feature":
                if model_name == "cmt":
                    feat_a, feat_b, labels = batch
                    feat_a, feat_b = feat_a.to(device), feat_b.to(device)
                    labels = labels.to(device)
                    logits, *rest = model(feat_a, feat_b)
                else:
                    feat_a, feat_b, labels = batch
                    feat_a, feat_b = feat_a.to(device), feat_b.to(device)
                    labels = labels.to(device)
                    logits, *rest = model(feat_a, feat_b)
            else:
                raise ValueError(f"mode must be 'raw' or 'feature', got: {mode}")

            loss         = criterion(logits, labels)
            total_loss   += loss.item()
            preds         = logits.argmax(dim=1)
            total_correct += (preds == labels).sum().item()

            for t, p in zip(labels.view(-1), preds.view(-1)):
                conf_matrix[t.long(), p.long()] += 1

            all_preds.extend(preds.cpu().numpy())
            all_logits.extend(logits.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            if rest:
                all_features.append(rest[0].cpu().numpy())

    n       = len(loader.dataset)
    elapsed = time.time() - t0

    return (
        total_loss / n,
        100.0 * total_correct / n,
        conf_matrix.numpy(),
        all_preds,
        all_logits,
        all_labels,
        all_features,
        elapsed / n,
    )


## 6. Utilities

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def count_parameters(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"  Parameters: {total:,} total  |  {trainable:,} trainable")
    return total, trainable


def sensitivity_specificity(conf):
    sens = conf[1, 1] / (conf[1, 1] + conf[1, 0])
    spec = conf[0, 0] / (conf[0, 0] + conf[0, 1])
    return sens, spec


def plot_training_curves(train_losses, val_losses, train_accs, val_accs):
    epochs = range(1, len(train_losses) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, train_losses, label="Train")
    ax1.plot(epochs, val_losses,   label="Val")
    ax1.set(title="Loss", xlabel="Epoch", ylabel="Loss")
    ax1.legend()

    ax2.plot(epochs, train_accs, label="Train")
    ax2.plot(epochs, val_accs,   label="Val")
    ax2.set(title="Accuracy (%)", xlabel="Epoch", ylabel="Accuracy")
    ax2.legend()

    plt.tight_layout()
    return fig


def plot_confusion_matrix(conf, sens, spec, class_names):
    fig, ax = plt.subplots(figsize=(7, 6))
    totals  = conf.sum(axis=1, keepdims=True)
    pct     = np.round(conf / totals * 100, 2)

    sns.heatmap(conf, annot=False, cmap="Blues",
                xticklabels=class_names, yticklabels=class_names, ax=ax)

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j + 0.5, i + 0.5,
                    f"{int(conf[i, j])}\n({pct[i, j]:.1f}%)",
                    ha="center", va="center", fontsize=10)

    ax.text(len(class_names) + 0.6, 0.5, f"Spec: {spec:.2f}",
            ha="center", va="center", fontsize=11, color="red", rotation=90)
    ax.text(len(class_names) + 0.6, 1.5, f"Sens: {sens:.2f}",
            ha="center", va="center", fontsize=11, color="red", rotation=90)

    ax.set(xlabel="Predicted", ylabel="True", title="Confusion Matrix")
    plt.tight_layout()
    return fig


def save_model(model, fold, out_dir, prefix):
    os.makedirs(out_dir, exist_ok=True)
    path = os.path.join(out_dir, f"{prefix}_fold{fold}.pth")
    torch.save(model.state_dict(), path)
    print(f"  Model saved → {path}")


def save_fold_results(fold, train_loss, train_acc, val_loss, val_acc,
                      conf, results_dir, inference_time=None, train_time=None):
    os.makedirs(results_dir, exist_ok=True)
    np.save(os.path.join(results_dir, f"conf_matrix_fold{fold}.npy"), conf)
    with open(os.path.join(results_dir, f"metrics_fold{fold}.txt"), "w") as fh:
        fh.write(f"Fold {fold}\n")
        fh.write(f"Train Loss: {train_loss:.4f}\n")
        fh.write(f"Train Acc:  {train_acc:.2f}%\n")
        fh.write(f"Val Loss:   {val_loss:.4f}\n")
        fh.write(f"Val Acc:    {val_acc:.2f}%\n")
        fh.write(f"Confusion Matrix:\n{conf}\n")
        if train_time:
            fh.write(f"Avg train time/epoch: {train_time:.4f}s\n")
        if inference_time:
            fh.write(f"Inference time/sample: {inference_time:.6f}s\n")


## 7. Main Pipeline



In [ ]:
def run_pipeline(cfg=None):
    cfg    = cfg or build_config()
    device = torch.device("cuda" if (torch.cuda.is_available() and cfg["use_cuda"]) else "cpu")
    print(f"Device: {device}")
    set_seed()

    # ── Base output directory ────────────────────────────────────────────────
    if cfg["model"] == "cmt":
        base_out = os.path.join(cfg["feature_dirs"]["CMT"], cfg["cv"])
    else:
        key      = f"{cfg['modality'].upper()}_{cfg['model'].upper()}"
        base_out = os.path.join(cfg["feature_dirs"][key], cfg["cv"])

    os.makedirs(base_out, exist_ok=True)

    # ── Load data ────────────────────────────────────────────────────────────
    with open(cfg["data_path"], "rb") as fh:
        df = pickle.load(fh)

    subject_ids  = df["subject_id"].values
    save_features = cfg["model"] in ("fcn", "alstm", "fat")

    # ── Cross-validation splitter ────────────────────────────────────────────
    if cfg["cv"] == "loso":
        splitter = LeaveOneGroupOut()
    else:
        splitter = GroupKFold(n_splits=10)

    splits = list(splitter.split(df, groups=subject_ids))

    # ── Fold loop ────────────────────────────────────────────────────────────
    for fold_idx, (train_idx, test_idx) in enumerate(splits):
        fold = fold_idx + 1

        if cfg["only_fold"] is not None and fold != cfg["only_fold"]:
            continue

        total = len(np.unique(subject_ids)) if cfg["cv"] == "loso" else 10
        print(f"\n{'='*60}")
        print(f"  Fold {fold}/{total}  |  model={cfg['model']}  |  cv={cfg['cv']}")
        print(f"{'='*60}")

        train_subj = np.unique(df["subject_id"].iloc[train_idx])
        test_subj  = np.unique(df["subject_id"].iloc[test_idx])
        print(f"  Train subjects: {train_subj}")
        print(f"  Test  subjects: {test_subj}")

        # ── Build datasets ───────────────────────────────────────────────────
        if cfg["model"] == "fat":
            mod   = cfg["modality"].upper()
            fcn_dir  = os.path.join(cfg["feature_dirs"][f"{mod}_FCN"],   cfg["cv"], "features")
            lstm_dir = os.path.join(cfg["feature_dirs"][f"{mod}_ALSTM"], cfg["cv"], "features")
            train_ds, test_ds = load_fusion_dataset(fcn_dir, lstm_dir, "fcn", "alstm", fold)
            mode = "feature"

        elif cfg["model"] == "cmt":
            eda_dir = os.path.join(cfg["feature_dirs"]["EDA_FAT"], cfg["cv"], "features")
            ecg_dir = os.path.join(cfg["feature_dirs"]["ECG_FAT"], cfg["cv"], "features")
            train_ds, test_ds = load_fusion_dataset(eda_dir, ecg_dir, "fat", "fat", fold)
            mode = "feature"

        else:
            train_ds, test_ds = prepare_raw_datasets(
                df, cfg["modality"], cfg["seq_len"],
                subject_ids[train_idx], subject_ids[test_idx]
            )
            mode = "raw"

        train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=False)
        test_loader  = DataLoader(test_ds,  batch_size=cfg["batch_size"], shuffle=False)

        # ── Model / optimiser / scheduler ────────────────────────────────────
        model     = build_model(cfg, device)
        count_parameters(model)

        optimizer = torch.optim.Adam(model.parameters(), lr=cfg["lr"])
        scheduler = torch.optim.lr_scheduler.MultiStepLR(
            optimizer, milestones=cfg["sched_milestones"], gamma=cfg["sched_gamma"]
        )
        criterion = nn.CrossEntropyLoss()

        # ── Training loop ────────────────────────────────────────────────────
        train_losses, val_losses   = [], []
        train_accs,   val_accs     = [], []
        t_start = time.time()

        for epoch in range(cfg["epochs"]):
            print(f"  Epoch {epoch+1}/{cfg['epochs']}")

            train_result = train_one_epoch(
                model, train_loader, optimizer, criterion,
                device, mode, cfg["model"], save_features
            )
            val_result = evaluate(
                model, test_loader, criterion,
                device, mode, cfg["model"]
            )

            t_loss, t_acc = train_result[:2]
            v_loss, v_acc, conf, _, _, val_labels, test_features, infer_t = val_result

            train_losses.append(t_loss)
            train_accs.append(t_acc)
            val_losses.append(v_loss)
            val_accs.append(v_acc)

            print(f"    train: loss={t_loss:.4f}  acc={t_acc:.2f}%")
            print(f"    val:   loss={v_loss:.4f}  acc={v_acc:.2f}%")
            scheduler.step()

        epoch_time = (time.time() - t_start) / cfg["epochs"]
        print(f"  Avg time/epoch: {epoch_time:.2f}s")

        # ── Save features (FCN / ALSTM / FAT only) ───────────────────────────
        if save_features and len(train_result) > 2:
            feat_out = os.path.join(base_out, "features")
            os.makedirs(feat_out, exist_ok=True)

            np.save(os.path.join(feat_out, f"{cfg['model']}_features_train_fold{fold}.npy"),
                    np.concatenate(train_result[2]))
            np.save(os.path.join(feat_out, f"labels_train_fold{fold}.npy"),
                    np.concatenate(train_result[3]))
            np.save(os.path.join(feat_out, f"{cfg['model']}_features_test_fold{fold}.npy"),
                    np.concatenate(test_features))
            np.save(os.path.join(feat_out, f"labels_test_fold{fold}.npy"),
                    np.array(val_labels))
            print(f"  Features saved → {feat_out}")

        # ── Persist results & plots ───────────────────────────────────────────
        results_dir = os.path.join(base_out, "results")
        save_fold_results(fold, t_loss, t_acc, v_loss, v_acc, conf,
                          results_dir, infer_t, epoch_time)
        save_model(model, fold, base_out, cfg["model"])

        fig = plot_training_curves(train_losses, val_losses, train_accs, val_accs)
        fig.savefig(os.path.join(results_dir, f"curves_fold{fold}.png"))
        plt.close(fig)

        sens, spec = sensitivity_specificity(conf)
        print(f"  Sensitivity: {sens:.3f}  |  Specificity: {spec:.3f}")

        fig = plot_confusion_matrix(conf, sens, spec, cfg["class_names"])
        fig.savefig(os.path.join(results_dir, f"conf_matrix_fold{fold}.png"))
        plt.close(fig)

    print("\nAll folds complete.")


## 8. Run


In [ ]:
run_pipeline()
